# tests to maunch the nifi starter

In [1]:
import openeo
import os
from time import sleep
import geopandas as gpd
from urllib.parse import quote
from shapely import to_geojson

In [2]:
# establish connection to OpenEO and authenticate
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [7]:
#namespace="/home/smetsb/PycharmProjects/OpenEO-UDP-UDF-catalogue/UDP/json/udp_eunis_mixer_alpha3.json"
#namespace="https://raw.githubusercontent.com/ESA-WEED-project/OpenEO-UDP-UDF-catalogue/refs/heads/alpha3_udp/UDP/json/udp_eunis_mixer_alpha3.json"
namespace="https://raw.githubusercontent.com/ESA-WEED-project/OpenEO-UDP-UDF-catalogue/refs/heads/UDP_trainstarter/UDP/json/udp_trainstarter.json"
#namespace="/home/deroob/Git/OpenEO-UDP-UDF-catalogue/UDP/json/udp_starter.json"

In [8]:
#define parameters
#param_bbox = {'west':520000, 'south':4260000,'east':540000,'north':4280000,'crs':32634}
param_name = "NL"
param_geometry = to_geojson(gpd.read_file("/data/users/Public/deroob/shape_NL.geosjon").geometry[0]) #geojson lite format of the area of interest
param_year = 2024
param_digitalId = "B1"
param_scenarioId = "v100"
param_rdm_table = 'global-training'
param_dt_url = "https://services.integratedmodelling.org/runtime/main/api/v1/dt/ESA_INSTITUTIONAL.rvr3s2juw0"

## get the habitat map

In [9]:
#get cube from udp
cube = connection.datacube_from_process(
    process_id="udp_trainstarter",
    namespace=namespace,
    geometry = param_geometry,
    year = param_year,
    drm_table = param_rdm_table,
    digitalId = param_digitalId,
    scenarioId = param_scenarioId,
    dt_url = param_dt_url
    )

In [10]:
print(cube.to_json())

{
  "process_graph": {
    "udpstarter1": {
      "process_id": "udp_starter",
      "arguments": {
        "bbox": {
          "west": 460000,
          "south": 4200000,
          "east": 480000,
          "north": 4220000,
          "crs": 32634
        },
        "digitalId": "B1",
        "dt_url": "https://services.integratedmodelling.org/runtime/main/api/v1/dt/ESA_INSTITUTIONAL.rvr3s2juw0",
        "onnx_model": "EUNIS2021plus_EU_v1_2024_MED",
        "scenarioId": "v100",
        "year": 2024
      },
      "namespace": "https://raw.githubusercontent.com/ESA-WEED-project/OpenEO-UDP-UDF-catalogue/refs/heads/main/UDP/json/udp_starter.json",
      "result": true
    }
  }
}


In [6]:
#create and start job
job = cube.create_job(title=f'UDP_tests_{param_digitalId}_{param_scenarioId}_AOI', auto_add_save_result=False)

Preflight process graph validation raised: [ProcessUnsupported] Process with identifier 'udp_starter' is not available in namespace '/home/deroob/Git/OpenEO-UDP-UDF-catalogue/UDP/json/udp_starter.json'.


In [11]:
job.start_job()

<BatchJob job_id='j-251203101931465bbc6a248395716c1c'>

In [12]:
#follow-up on job
print(job.job_id)
while job.status() not in ['finished','error','canceled']:
    print(f"going to sleep job not yet done: status : {job.status()}")
    sleep(10)

print(f"Job done: status : {job.status()}")

j-251203101931465bbc6a248395716c1c
going to sleep job not yet done: status : created
going to sleep job not yet done: status : running
going to sleep job not yet done: status : running
going to sleep job not yet done: status : running
going to sleep job not yet done: status : running
going to sleep job not yet done: status : running
going to sleep job not yet done: status : running
going to sleep job not yet done: status : running
going to sleep job not yet done: status : running
Job done: status : finished


In [13]:
#get results (metadata)

job.get_results()

<JobResults for job 'j-251203101931465bbc6a248395716c1c'>

In [14]:
#get errors
job.logs(level='error')

[]